# 朴素贝叶斯

In [1]:
def loadDataSet():
    postingList = [['my', 'dog', 'has', 'flea', \
                    'problems', 'help', 'please'],
                    ['maybe', 'not', 'take', 'him', \
                    'to', 'dog', 'park', 'stupid'],
                    ['my', 'dalmation', 'is', 'so', 'cute', \
                    'I', 'love', 'him'],
                    ['stop', 'posting', 'stupid', 'worthless', 'garbage'],
                    ['mr', 'licks', 'ate', 'my', 'steak', 'how',\
                    'to', 'stop', 'him'],
                    ['quit', 'buying', 'worthless', 'dog', 'food', 'stupid']]
    classVec = [0, 1, 0, 1, 0, 1]
    return postingList, classVec

# 创建一个含有数据集中所有单词的列表
def createVocabList(dataSet: list) -> list:
    vocabSet = set([])
    for document in dataSet:
        vocabSet = vocabSet | set(document) # |取并集
    return list(vocabSet)

def setOfWords2Vec(vocabList: list[str], inputSet: set[str]) -> list[int]:
    returnVec = [0]*len(vocabList)
    for word in inputSet:
        if word in vocabList:
            returnVec[vocabList.index(word)] = 1
        else:
            print(f'the word: {word} is not in my Vocabulary!')
    return returnVec

In [2]:
listOPosts, listClasses = loadDataSet() # 文章, 特征向量
myVocabList = createVocabList(listOPosts)
for Vocab in myVocabList:
    print(Vocab, end=' ')

problems my stop love so help dog stupid is posting licks steak park cute quit take food him please I has dalmation ate flea worthless mr how maybe garbage buying not to 

In [3]:
Vec = setOfWords2Vec(myVocabList, listOPosts[0])
print(Vec, end=' ')

[1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0] 

## 从词向量到计算概率

$$p(c_i | w) = \frac{p(w|c_i)p(c_i)}{p(w)}$$

其中w为一个向量。  

对每个类计算该值，再比较这两个概率值大小。  
通过类别i中文档数除以总的督文档数计算（如classVec=[0,1,0,1,0,1]中1的数目除以总的数目）计算$p(w|c_i)$  
计算$p(w|c_i)$，使用朴素贝叶斯假设，假设所有次互相独立，将w展开为一个个独立特征，那么可将$p(w|c_i)$写作$p(w_0,w_1,w_2,...,w_n|c_i)$  
那么可以将使用$p(w_0|ci)p(w_1|c_i)...p(w_N|c_i)$来计算上述概率

伪代码如下：
```markdown
计算每个类别中的文档数目
对每篇训练文档：
    对每个类别：
        如果词条出现在文档中：增加该词条计数值
        增加所有词条计数值
对每个类别：
    对每个词条：
        将该词条的数目除以总词条数目得到条件概率
返回每个类别的条件概率
```

In [4]:
import numpy as np
def trainNB0(trainMatrix:list[list[int]], trainCategory:list[int]) -> tuple[np.ndarray, np.ndarray, float]:
    numTrainDocs = len(trainMatrix) # 矩阵每行有多少元素
    numWords = len(trainMatrix[0])  # 矩阵每列有多少元素
    pAbusive = sum(trainCategory) / float(numTrainDocs) # 是辱骂意思的概率
    p0Num, p1Num = np.zeros(numWords), np.zeros(numWords)
    p0Denom, p1Denom = 0.0, 0.0
    for i in range(numTrainDocs):
        if trainCategory[i] == 1:
            p1Num += trainMatrix[i] # 设置的向量加词频向量，可以得到类别为1的词的出现频率
            p1Denom += sum(trainMatrix[i]) # 计算文档词频的和
        else:
            p0Num += trainMatrix[i]
            p0Denom += sum(trainMatrix[i])
    p1Vect = p1Num/p1Denom # 计算 向量除以词频之和，向量中得出的就是坏词概率
    p0Vect = p0Num/p0Denom
    return p0Vect, p1Vect, pAbusive

In [5]:
listOPosts, listClasses = loadDataSet()
myVocabList = createVocabList(listOPosts)
trainMat = []
for postinDoc in listOPosts:
    trainMat.append(setOfWords2Vec(myVocabList, postinDoc))

p0V, p1V, pAb = trainNB0(trainMat, listClasses)

print(p0V, end=' ')
print()
print(p1V, end=' ')
print()
print(pAb, end=' ')

[0.04166667 0.125      0.04166667 0.04166667 0.04166667 0.04166667
 0.04166667 0.         0.04166667 0.         0.04166667 0.04166667
 0.         0.04166667 0.         0.         0.         0.08333333
 0.04166667 0.04166667 0.04166667 0.04166667 0.04166667 0.04166667
 0.         0.04166667 0.04166667 0.         0.         0.
 0.         0.04166667] 
[0.         0.         0.05263158 0.         0.         0.
 0.10526316 0.15789474 0.         0.05263158 0.         0.
 0.05263158 0.         0.05263158 0.05263158 0.05263158 0.05263158
 0.         0.         0.         0.         0.         0.
 0.10526316 0.         0.         0.05263158 0.05263158 0.05263158
 0.05263158 0.05263158] 
0.5 

- 计算$p(w_0|ci)p(w_1|c_i)...p(w_N|c_i)$时，有一个概率值为0，则最后乘积为0。  
改善这种影响，可以将所有词的出现次数初始化为1，并将分母初始化为2。
- 太多极小数相乘造成下溢出  
采用$ln(a*b) = ln(a) + ln(b)$

In [ ]:
import numpy as np
def trainNB0(trainMatrix:list[list[int]], trainCategory:list[int]) -> tuple[np.ndarray, np.ndarray, float]:
    numTrainDocs = len(trainMatrix) 
    numWords = len(trainMatrix[0])  
    pAbusive = sum(trainCategory) / float(numTrainDocs)
    p0Num, p1Num = np.ones(numWords), np.ones(numWords) # 修改
    p0Denom, p1Denom = 2.0, 2.0 # 修改
    for i in range(numTrainDocs):
        if trainCategory[i] == 1:
            p1Num += trainMatrix[i]
            p1Denom += sum(trainMatrix[i])
        else:
            p0Num += trainMatrix[i]
            p0Denom += sum(trainMatrix[i])
    p1Vect = np.log(p1Num/p1Denom) # 修改
    p0Vect = np.log(p0Num/p0Denom) # 修改
    return p0Vect, p1Vect, pAbusive

In [6]:
def classifyNB(vec2Classify: np.ndarray, p0Vec: np.ndarray, p1Vec: np.ndarray, pClass1: float) -> int:
    p1 = sum(vec2Classify * p1Vec) + np.log(pClass1)
    p0 = sum(vec2Classify * p0Vec) + np.log(1.0 - pClass1)
    if p1 > p0:
        return 1
    else:
        return 0
    
def testingNB():
    listOPosts, listClasses = loadDataSet() # 加载原始数据和标签
    myVocabList = createVocabList(listOPosts) # 创建词汇表

    trainMat = [] # 将每个训练文档转换成词集向量
    for postinDoc in listOPosts:
        trainMat.append(setOfWords2Vec(myVocabList, postinDoc))

    p0V, p1V, pAb = trainNB0(np.array(trainMat), np.array(listClasses))
    testEntry =['love', 'my', 'dalmation']
    thisDoc = np.array(setOfWords2Vec(myVocabList, testEntry))
    print(f'{testEntry} classified as: {classifyNB(thisDoc, p0V, p1V, pAb)}')
    testEntry = ['stupid', 'garbage']
    thisDoc = np.array(setOfWords2Vec(myVocabList, testEntry))
    print(f'{testEntry} classified as: {classifyNB(thisDoc, p0V, p1V, pAb)}')

In [7]:
testingNB()

['love', 'my', 'dalmation'] classified as: 0
['stupid', 'garbage'] classified as: 1


In [ ]:
def bagOfWords2VecMN(vocabList, inputSet):
    returnVec = [0]*len(vocabList)
    for word in inputSet:
        if word in inputSet:
            returnVec[vocabList.index(word)] += 1
        return returnVec